In [1]:
import json
from pathlib import Path
import pandas as pd
import re
from scripts import extract as ext

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [3]:
with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/temas_ejercicio.json", "r") as file:
    temas_ejercicio = json.load(file)

with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/quimica_keywords.json") as file:
    quimica_keywords = json.load(file)

In [4]:
def classify_exercise(exam_name: str, statement: str, keywords = quimica_keywords) -> tuple:
    # obtener el año del examen
    exam_year = int(exam_name.split("_")[0])
    exam_name = (" - ".join(exam_name.split("_")[1:])).capitalize()

    # seleccionar modelo de examen en función del año
    if exam_year < 2020:
        exam_model = "modelo_2010_2019"
        exercise_number_pattern = r"^\d{1}\.-"

    elif exam_year < 2024:
        exam_model = "modelo_2020_2023"
        exercise_number_pattern = r"^[ABC]\d{1}"
    else:
        exam_model = "modelo_2024+"
        exercise_number_pattern = r"^\d{1}?[AB]"

    # temas del ejercicio según número ejercicio y modelo

    # "modelo_2024+": {
    #     "tipo_numeración" : "1A, 1B, 2A, 2B, ..., 4B, 5",
    #     "1": ["configuracion_electronica", ...]
    #     "2": ["equilibrio_quimico", ...],

    exercise_topics_dict = temas_ejercicio[exam_model]

    # obtener el número del ejercicio
    exercise_number = re.match(exercise_number_pattern, statement)

    # si se encuentra el número de ejercicio
    if exercise_number:

        # extraer la clave del diccionario del modelo de examen
        if exam_year < 2020:  # 2010 - 2019 - key 1,2,3,...
            exercise_number_key = exercise_number[0][0]

            # las keys son: "1", "2,3,4", "5,6"
            # exercise_number_key necesita un ajuste
            # para que se encuentre la clave
            for key in exercise_topics_dict.keys():
                if exercise_number_key in key:
                    exercise_number_key = key
                    break
        
        elif 2020 <= exam_year <= 2023:  # 2020 - 2023 - key A, B, C
            exercise_number_key = exercise_number[0][0]

        else:  # 2024 + - key 1, 2, 3, 4A, 4B, 5
            exercise_number_key = exercise_number[0]
            if exercise_number[0] not in  ["4A", "4B"]:
                exercise_number_key = exercise_number[0][0]
            else:
                exercise_number_key = exercise_number[0]

        # limpiar el número de ejercicio
        exercise_number = re.sub(r"[\.-]", "", exercise_number[0])

        # ejercicios de formulación - NO CLASIFICAR

        # condiciones a cumplir para ser ejercicio de formulación
        formulation_pre_2020 = exam_year <= 2019 and exercise_number_key == "1"
        formulation_2020_2023 =  (2020 <= exam_year <= 2023) and exercise_number_key == "A"

        if formulation_pre_2020 or formulation_2020_2023:
            exercise_topic = "Formulación"
            exercise_types = ["Formulación"]

        elif exam_year >= 2024 and exercise_number_key == "4A":
            exercise_topic = "Formulación"
            exercise_types = ["Formulación orgánica"]

        else:

            ###### CLASIFICACIÓN #######

            # opciones de tema para el ejercicio suministrado
            exercise_topic_options = exercise_topics_dict[exercise_number_key]

            # recorro los posibles temas del ejercicio
            # e intento clasificar el tema del ejercicio
            exercise_topic = ""
            for topic in exercise_topic_options:
                
                # obtengo las palabras clave que identifican el tema
                # y los tipos de ejercicios del tema y sus respectivas keywords
                topic_identifying_keywords = keywords[topic]["keywords_tema"]
                for kw in topic_identifying_keywords:
                    if kw in statement:
                        exercise_topic = topic
                        break

                # si se ha clasificado el ejercicio deja de buscar
                if exercise_topic:
                    break

            # lista de tipos de ejercicio
            exercise_types = []

            # si se ha identificado el tema
            # lo siguiente es asignar tipo(s) de ejercicio
            if exercise_topic:
                # compruebo existencia de keywords de cada tipo de ejercicio en el enunciado
                exercise_type_keywords = keywords[exercise_topic]["tipos"]
                for t, kw_list in exercise_type_keywords.items():
                    # comprueba lista de keywords para ese tipo de ejercicio
                    # con que haga match una keyword con el enunciado
                    # clasifícalo com ese tipo de ejercicio
                    if any(kw in statement for kw in kw_list):
                        exercise_types.append(t)
            else:
                exercise_topic, exercise_types = None, [] 
    else:
        exercise_topic, exercise_types = None, []

    return (exam_year, exam_name, exercise_number, exercise_topic, exercise_types, statement)

In [5]:
file = ROOT / "examenes_quimica/2013_quimica/2013_quimica_septiembre_reserva_a.pdf"

In [6]:
exam_text = ext.extract_exercises_from_pdf(file)

exam_year = int(file.name.split("_")[0])

if exam_year < 2020:
    exam_period = "2010-2019"
elif exam_year <= 2024:
    exam_period = "2020-2024"
else:
    exam_period = "2025+"

exercises_list = ext.split_exercises(exam_text, exam_period=exam_period)

In [7]:
print(exercises_list[11][1])

6.- AI pasar una corriente durante el tiempo de una hora y cincuenta minutos a través de una disolución
de Cu(II), se depositan 1,82 g de cobre.
a) Calcule la intensidad de la corriente que ha circulado.
b) Calcule la carga del electrón.
Datos: F = 96500 C. Masa atómica Cu = 63,5.


In [9]:
classify_exercise(file.stem, exercises_list[11][1], quimica_keywords)

(2013,
 'Quimica - septiembre - reserva - a',
 '6',
 'redox',
 ['electrolisis'],
 '6.- AI pasar una corriente durante el tiempo de una hora y cincuenta minutos a través de una disolución\nde Cu(II), se depositan 1,82 g de cobre.\na) Calcule la intensidad de la corriente que ha circulado.\nb) Calcule la carga del electrón.\nDatos: F = 96500 C. Masa atómica Cu = 63,5.')

In [12]:
year_classifications = []
for file in (ROOT / "examenes_quimica/2012_quimica").iterdir():
    if file.is_file():
        exam_text = ext.extract_exercises_from_pdf(file)
        exam_year = int(file.name.split("_")[0])

        if exam_year < 2020:
            exam_period = "2010-2019"
        elif exam_year <= 2024:
            exam_period = "2020-2024"
        else:
            exam_period = "2025+"

        exercises_list = ext.split_exercises(exam_text, exam_period=exam_period)

        # exercises_list devuelve una tupla: (exam_option, statement)
        for exercise_tuple in exercises_list:
            try:
                statement = exercise_tuple[1]
                exam_option = exercise_tuple[0]
                classification = classify_exercise(file.stem, statement, keywords=quimica_keywords)
                classification = list(classification)
                classification.append(exam_option)
                year_classifications.append(classification)
            except Exception as e:
                print(f"Error: file {file.name}. Exception: {e}")

df = pd.DataFrame(year_classifications, columns = ["year", "exam_name", "exercise_number", "topic", "exercise_type", "statement", "exam_option"])
df = df.explode("exercise_type", ignore_index=True)

In [13]:
df

,year,exam_name,exercise_number,topic,exercise_type,statement,exam_option
0,2012,Quimica - 2 - modelo - 5b,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,"[Opción A, Opción A, Opción A, Opción A, Opció..."
1,2012,Quimica - 2 - modelo - 5b,2,NaN,NaN,2.- Razone si son verdaderas o falsas las sigu...,"[Opción A, Opción A, Opción A, Opción A, Opció..."
2,2012,Quimica - 2 - modelo - 5b,3,enlace_químico,Polaridad,3.- La notación de una pila electroquímica es:...,"[Opción A, Opción A, Opción A, Opción A, Opció..."
3,2012,Quimica - 2 - modelo - 5b,4,ácido_base,NaN,"4.- Dadas las siguientes especies químicas, en...","[Opción A, Opción A, Opción A, Opción A, Opció..."
4,2012,Quimica - 2 - modelo - 5b,5,termoquímica,Relación entalpía y energía interna,5.- Las entalpías estándar de combustión a 25 ...,"[Opción A, Opción A, Opción A, Opción A, Opció..."
...,...,...,...,...,...,...,...
121,2012,Quimica - 6 - modelo - 4,3,NaN,NaN,"3.- Considerando condiciones estándar a 25 ºC,...","[Opción A, Opción A, Opción A, Opción A, Opció..."
122,2012,Quimica - 6 - modelo - 4,4,reactividad_orgánica,Isomería,4.- Escriba la fórmula desarrollada de:\na) Do...,"[Opción A, Opción A, Opción A, Opción A, Opció..."
123,2012,Quimica - 6 - modelo - 4,5,termoquímica,NaN,"5.- La reacción de la hidracina, N₂H₄, con el ...","[Opción A, Opción A, Opción A, Opción A, Opció..."
124,2012,Quimica - 6 - modelo - 4,6,solubilidad,formación de precipitado,6.- En diversos países la fluoración del agua ...,"[Opción A, Opción A, Opción A, Opción A, Opció..."


In [ ]:
df[df.topic.isna()]

,year,exam_name,exercise_number,topic,exercise_type,statement


In [ ]:
for st in df[df.topic.isna()].statement:
    print(st)

In [ ]:
df.shape

(123, 5)